# NB-final — consistency ablation + test-set results + figures

Two phases:
1. **Train** `segformer_noconsist` (3 seeds) — the one clean ablation isolating the
   consistency loss on the final backbone. (`segformer_concat` is already trained.)
2. **Evaluate** the final model + baselines on the **TEST** split (selected on val,
   reported on test) and generate: the results table, the **degradation curve**,
   and the **params-vs-IoU Pareto** plot.

**Attach:** `lgmf_c` (v7), Sen1Floods11, `lgmf-index`, `lgmf-segf` (the segformer_confirm
output), `lgmf-baselines` (NB3 output). GPU + Internet on. ~2h (3 train + eval).


In [ ]:
import sys, os, glob, json, random, time, dataclasses, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v7"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
try:
    import segmentation_models_pytorch  # noqa
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.baselines import build_baseline, build_gated_unet
from lgmf.model import build_model, count_params
from lgmf.metrics import flops_g
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__)
idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"; regs = ["both", "s1_only", "s2_only"]
os.makedirs("/kaggle/working/figs", exist_ok=True)

## Phase 1 — train segformer_noconsist (3 seeds)

In [ ]:
SEEDS, EPOCHS = [42, 1, 2], 45
def make_cfg(seed):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size = EPOCHS, 3e-4, 8
    c.ema_decay, c.loss_type = 0.99, "focal_tversky"
    c.consistency_weight = 0.0                        # <-- ablation: no consistency
    c.num_workers = 0
    c.split_dir, c.ckpt_dir = idx, "/kaggle/working/ckpt_final"
    c.s1_mean, c.s1_std = stats["s1_mean"], stats["s1_std"]
    c.s2_mean, c.s2_std = stats["s2_mean"], stats["s2_std"]
    c.seed, c.run_name = seed, "segformer_noconsist"
    return c

def seed_all(sd):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False

def train_noconsist(cfg):
    seed_all(cfg.seed)
    tr = DataLoader(Sen1Floods11(cfg, f"{idx}/train_index.json", augment=True),
                    batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=pin, drop_last=True)
    va = DataLoader(Sen1Floods11(cfg, f"{idx}/val_index.json", augment=False),
                    batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    model = build_baseline(arch="segformer", encoder="mit_b0", in_ch=8, pretrained=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type=="cuda") else None
    ema = ModelEMA(model, cfg.ema_decay); em = ema.module
    best = -1.0
    for epoch in range(cfg.epochs):
        train_one_epoch(model, tr, opt, cfg, epoch, device, scaler=scaler, ema=ema)
        sched.step()
        val = evaluate(em, va, cfg, device)
        sc = float(np.mean([val[r]["iou"] for r in regs]))
        if sc > best:
            best = sc
            D = os.path.join(cfg.ckpt_dir, cfg.run_name, f"seed{cfg.seed}"); os.makedirs(D, exist_ok=True)
            torch.save({"model": model.state_dict(), "ema": ema.module.state_dict(),
                        "baseline": {"in_ch": 8, "arch": "segformer", "encoder": "mit_b0"},
                        "cfg": dict(cfg.__dict__)}, os.path.join(D, "best.pt"))
    return best

t0 = time.time()
for sd in SEEDS:
    D = os.path.join("/kaggle/working/ckpt_final", "segformer_noconsist", f"seed{sd}")
    if os.path.exists(os.path.join(D, "best.pt")):
        print(f"[skip] noconsist seed{sd}"); continue
    b = train_noconsist(make_cfg(sd))
    print(f"[done] noconsist seed{sd} val-mean {b:.3f}  ({(time.time()-t0)/60:.0f} min)")

## Phase 2 — evaluate everything on the TEST split

In [ ]:
FIELDS = {f.name for f in dataclasses.fields(LGMFConfig)}
def build_from_ckpt(ck):
    if ck.get("gated"):
        kw = {k: v for k, v in ck["gated"].items() if k != "out_ch"}
        m = build_gated_unet(pretrained=False, out_ch=ck["gated"].get("out_ch", 8), **kw)
    elif ck.get("baseline"):
        kw = {k: v for k, v in ck["baseline"].items() if k != "in_ch"}
        m = build_baseline(pretrained=False, in_ch=ck["baseline"].get("in_ch", 8), **kw)
    else:
        c = LGMFConfig(**{k: v for k, v in ck["cfg"].items() if k in FIELDS})
        c.s2_bands = tuple(c.s2_bands); c.encoder_pretrained = False
        m = build_model(c)
    m.load_state_dict(ck.get("ema") or ck["model"]); return m

def mkloader(split):
    base = LGMFConfig(); base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]
    base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]; base.num_workers = 0
    ds = Sen1Floods11(base, f"{idx}/{split}_index.json", augment=False)
    return DataLoader(ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=pin)
test_ld, bol_ld = mkloader("test"), mkloader("bolivia")

# which configs to report, and a display label
REPORT = {
    "segformer_concat":    "SegFormer + curric + consist (ours)",
    "segformer_noconsist": "SegFormer + curric (no consist)",
    "unet_drop":           "U-Net + curric (concat)",
    "manet_drop":          "MANet + curric (concat)",
    "unet_naive":          "U-Net (no curric)",
}
all_ck = glob.glob("/kaggle/input/**/best.pt", recursive=True) + glob.glob("/kaggle/working/**/best.pt", recursive=True)
def parse(p):
    sd = os.path.dirname(p); return os.path.basename(os.path.dirname(sd)), os.path.basename(sd)

data = {}       # config -> list of per-seed dicts (test + bol)
meta = {}       # config -> (params, gflops)
for p in sorted(set(all_ck)):
    cfg, seed = parse(p)
    if cfg not in REPORT: continue
    ck = torch.load(p, map_location=device, weights_only=False)
    m = build_from_ckpt(ck).to(device).eval()
    if cfg not in meta:
        meta[cfg] = (count_params(m), flops_g(m, LGMFConfig(), device="cpu"))
    te = evaluate(m, test_ld, LGMFConfig(), device)
    bo = evaluate(m, bol_ld, LGMFConfig(), device)
    pack = lambda res: {r: {"iou": res[r]["iou"], "f1": res[r]["f1"],
                            "flood": res[r].get("flood_only_iou")} for r in regs}
    data.setdefault(cfg, []).append({"test": pack(te), "bol": pack(bo)})
    print(f"  {cfg:20s} {seed:6s} test both {te['both']['iou']:.3f} s1 {te['s1_only']['iou']:.3f}")
print("evaluated:", {k: len(v) for k, v in data.items()})

## Main results table (TEST, mean ± std)

In [ ]:
def agg(cfg, split, reg, key="iou"):
    a = np.array([r[split][reg][key] for r in data[cfg] if r[split][reg][key] is not None])
    return (a.mean(), a.std()) if len(a) else (float("nan"), 0.0)
def c(t): return f"{t[0]:.3f}±{t[1]:.3f}"
order = [k for k in REPORT if k in data]

print("=" * 108)
print(f"{'model':38s} {'params':>7s} {'GFLOPs':>7s} {'both':>12s} {'SAR-only':>12s} {'optical':>12s} {'flood-only':>12s}")
print("-" * 108)
for k in order:
    p, g = meta[k]
    print(f"{REPORT[k]:38s} {p:6.2f}M {g:7.1f} {c(agg(k,'test','both')):>12s} "
          f"{c(agg(k,'test','s1_only')):>12s} {c(agg(k,'test','s2_only')):>12s} "
          f"{c(agg(k,'test','s1_only','flood')):>12s}")
print("=" * 108)

# consistency ablation callout
if "segformer_concat" in data and "segformer_noconsist" in data:
    dc = agg("segformer_concat","test","s1_only")[0] - agg("segformer_noconsist","test","s1_only")[0]
    print(f"\nConsistency effect (test SAR-only): {dc:+.3f}  "
          f"[{'confirmed' if dc>0 else 'not confirmed'}]")

json.dump({"data": data, "meta": meta}, open("/kaggle/working/final_results.json","w"), indent=2, default=str)

## Figure — degradation curve (headline)

In [ ]:
labels = ["Both", "Optical-only", "SAR-only"]
key = ["both", "s2_only", "s1_only"]
fig, ax = plt.subplots(figsize=(5.2, 3.8))
for k in order:
    y = [agg(k, "test", r)[0] for r in key]
    e = [agg(k, "test", r)[1] for r in key]
    lw = 2.4 if k == "segformer_concat" else 1.3
    ax.errorbar(labels, y, yerr=e, marker="o", linewidth=lw, capsize=3,
                label=REPORT[k].replace(" (ours)", ""))
ax.set_ylabel("Test IoU"); ax.set_xlabel("Modality availability")
ax.grid(alpha=0.3); ax.legend(fontsize=7)
ax.set_title("Graceful degradation under missing modalities")
for ext in ("png", "pdf"):
    fig.savefig(f"/kaggle/working/figs/degradation.{ext}", dpi=300, bbox_inches="tight")
plt.show(); print("saved figs/degradation.pdf")

## Figure — efficiency Pareto (params vs IoU)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.8))
for k in order:
    p, _ = meta[k]; y = agg(k, "test", "both")[0]; ys = agg(k, "test", "s1_only")[0]
    is_ours = k == "segformer_concat"
    ax.scatter(p, y, s=90 if is_ours else 55, marker="*" if is_ours else "o",
               zorder=3, label=REPORT[k].replace(" (ours)", ""))
    ax.annotate(f"{REPORT[k].split(' +')[0].split(' (')[0]}", (p, y),
                textcoords="offset points", xytext=(6, 5), fontsize=7)
ax.set_xlabel("Parameters (M)"); ax.set_ylabel("Test IoU (both modalities)")
ax.grid(alpha=0.3); ax.set_title("Accuracy vs model size (top-left is better)")
for ext in ("png", "pdf"):
    fig.savefig(f"/kaggle/working/figs/pareto.{ext}", dpi=300, bbox_inches="tight")
plt.show(); print("saved figs/pareto.pdf")

## Done — you have your results

- `final_results.json`, `figs/degradation.pdf`, `figs/pareto.pdf`
- Report **test** numbers; Bolivia is in the json as out-of-region generalization.
- Headline model: **SegFormer + curriculum + consistency (3.7M)**.

Next is writing, not running. The story:
1. Curriculum is essential (unet_naive vs the curric models).
2. Consistency adds SAR-only robustness (the ablation above).
3. Lightweight SegFormer backbone matches/beats heavier CNNs at ~half the params.
4. Specialized fusion modules don't help (your 4 prior experiments = ablation appendix).
